In [ ]:
# Basic NumPy array serialization and deserialization with dtype preservation
import msgpack
import msgpack_numpy as m
import numpy as np
from typing import Any, Dict, List

print("=== msgpack-numpy Basic Array Serialization ===")

# Enable msgpack-numpy hooks
m.patch()

def demonstrate_basic_serialization():
    """Demonstrate basic NumPy array serialization/deserialization"""
    
    # Create various NumPy arrays with different dtypes
    arrays = {
        'int32_array': np.array([1, 2, 3, 4, 5], dtype=np.int32),
        'float64_array': np.array([1.1, 2.2, 3.3, 4.4], dtype=np.float64),
        'complex_array': np.array([1+2j, 3+4j, 5+6j], dtype=np.complex128),
        'bool_array': np.array([True, False, True, False], dtype=bool),
        'string_array': np.array(['hello', 'world', 'numpy'], dtype='U10'),
        # datetime64 is unsupported by msgpack-numpy 0.4.8 (dtype 'M' has no
        # buffer-protocol format -> ValueError on pack; upstream gap). A
        # non-native-byte-order array probes dtype/endianness preservation instead.
        'bigendian_array': np.array([1.5, -2.25, 3.125], dtype='>f8')
    }
    
    results = {}
    
    for name, array in arrays.items():
        # Serialize the array
        serialized = msgpack.packb(array)
        
        # Deserialize the array
        deserialized = msgpack.unpackb(serialized, strict_map_key=False)
        
        # Verify preservation of data and metadata
        data_equal = np.array_equal(array, deserialized)
        dtype_preserved = array.dtype == deserialized.dtype
        shape_preserved = array.shape == deserialized.shape
        
        results[name] = {
            'original_dtype': str(array.dtype),
            'original_shape': array.shape,
            'serialized_size': len(serialized),
            'deserialized_dtype': str(deserialized.dtype),
            'deserialized_shape': deserialized.shape,
            'data_equal': data_equal,
            'dtype_preserved': dtype_preserved,
            'shape_preserved': shape_preserved,
            'compression_ratio': array.nbytes / len(serialized) if len(serialized) > 0 else 0
        }
    
    return results

# Test basic serialization
serialization_results = demonstrate_basic_serialization()

print("Basic Serialization Results:")
for name, result in serialization_results.items():
    print(f"\n{name}:")
    print(f"  Original: {result['original_dtype']} {result['original_shape']}")
    print(f"  Preserved: dtype={result['dtype_preserved']}, shape={result['shape_preserved']}, data={result['data_equal']}")
    print(f"  Serialized size: {result['serialized_size']} bytes")
    print(f"  Compression ratio: {result['compression_ratio']:.2f}x")

# Demonstrate nested data structures with NumPy arrays
nested_data = {
    'metadata': {'version': '1.0', 'created': '2023-01-01'},
    'arrays': {
        'features': np.random.rand(100, 10).astype(np.float32),
        'labels': np.random.randint(0, 5, 100).astype(np.int8),
        'weights': np.ones(100).astype(np.float64)
    },
    'statistics': {
        'mean': np.array([1.2, 3.4, 5.6]),
        'std': np.array([0.5, 0.8, 1.1])
    }
}

# Serialize complex nested structure
print(f"\nNested Structure Serialization:")
nested_serialized = msgpack.packb(nested_data)
nested_deserialized = msgpack.unpackb(nested_serialized, strict_map_key=False)

print(f"Original structure keys: {list(nested_data.keys())}")
print(f"Deserialized structure keys: {list(nested_deserialized.keys())}")
print(f"Features array shape: {nested_deserialized['arrays']['features'].shape}")
print(f"Features array dtype: {nested_deserialized['arrays']['features'].dtype}")
print(f"Serialized size: {len(nested_serialized):,} bytes")

print("Basic NumPy array serialization completed successfully!")

In [ ]:
# Complex array structures and multi-dimensional data handling
import time
from typing import Tuple, Union

print("=== msgpack-numpy Complex Array Structures ===")

def create_complex_array_structures():
    """Create various complex NumPy array structures for testing"""
    
    structures = {}
    
    # 1. Multi-dimensional arrays with different shapes
    structures['tensor_3d'] = np.random.rand(50, 20, 10).astype(np.float32)
    structures['tensor_4d'] = np.random.rand(10, 8, 6, 4).astype(np.float16)
    structures['tensor_5d'] = np.random.rand(5, 4, 3, 2, 2).astype(np.int16)
    
    # 2. Structured arrays (like database records)
    dt = np.dtype([
        ('id', 'i4'),
        ('name', 'U20'),
        ('score', 'f8'),
        ('active', '?'),
        ('data', 'f4', (3,))
    ])
    
    structured_data = np.array([
        (1, 'Alice', 95.5, True, [1.1, 2.2, 3.3]),
        (2, 'Bob', 87.2, False, [4.4, 5.5, 6.6]),
        (3, 'Carol', 92.1, True, [7.7, 8.8, 9.9])
    ], dtype=dt)
    
    structures['structured_array'] = structured_data
    
    # 3. Sparse-like structures (many zeros)
    sparse_array = np.zeros((100, 100))
    sparse_array[10:20, 30:40] = np.random.rand(10, 10)
    sparse_array[50:60, 70:80] = np.random.rand(10, 10)
    structures['sparse_array'] = sparse_array.astype(np.float32)
    
    # 4. Memory views and advanced indexing
    base_array = np.arange(1000).reshape(10, 10, 10)
    structures['memory_view'] = base_array[::2, ::2, ::2]  # Strided view
    structures['boolean_indexed'] = base_array[base_array > 500]
    
    # 5. Record arrays with mixed types
    record_array = np.rec.fromarrays([
        np.arange(100),
        np.random.rand(100),
        np.random.choice(['A', 'B', 'C'], 100)
    ], names=['index', 'value', 'category'])
    
    structures['record_array'] = record_array
    
    return structures

def serialize_complex_structures(structures: Dict[str, np.ndarray]) -> Dict[str, Dict]:
    """Serialize complex array structures and analyze results"""
    
    results = {}
    
    for name, array in structures.items():
        print(f"Processing {name}...")
        
        try:
            # Serialize
            start_time = time.time()
            serialized = msgpack.packb(array)
            serialize_time = time.time() - start_time
            
            # Deserialize
            start_time = time.time()
            deserialized = msgpack.unpackb(serialized, strict_map_key=False)
            deserialize_time = time.time() - start_time
            
            # Analyze results
            original_memory = array.nbytes
            serialized_size = len(serialized)
            compression_ratio = original_memory / serialized_size
            
            # Verify integrity
            arrays_equal = np.array_equal(array, deserialized)
            dtype_match = array.dtype == deserialized.dtype
            shape_match = array.shape == deserialized.shape
            
            results[name] = {
                'original_shape': array.shape,
                'original_dtype': str(array.dtype),
                'original_memory': original_memory,
                'serialized_size': serialized_size,
                'compression_ratio': compression_ratio,
                'serialize_time': serialize_time,
                'deserialize_time': deserialize_time,
                'arrays_equal': arrays_equal,
                'dtype_match': dtype_match,
                'shape_match': shape_match,
                'integrity_check': arrays_equal and dtype_match and shape_match
            }
            
        except Exception as e:
            results[name] = {
                'error': str(e),
                'original_shape': array.shape,
                'original_dtype': str(array.dtype)
            }
    
    return results

# Create and test complex structures
complex_structures = create_complex_array_structures()
serialization_results = serialize_complex_structures(complex_structures)

print(f"\nComplex Structure Serialization Results:")
print(f"{'Structure':<20} {'Shape':<15} {'Dtype':<10} {'Size(MB)':<10} {'Compress':<8} {'Time(ms)':<10} {'Valid':<6}")
print("-" * 85)

for name, result in serialization_results.items():
    if 'error' not in result:
        shape_str = f"{result['original_shape']}"[:14]
        dtype_str = str(result['original_dtype'])[:9]
        size_mb = result['serialized_size'] / (1024 * 1024)
        compress_ratio = result['compression_ratio']
        total_time = (result['serialize_time'] + result['deserialize_time']) * 1000
        valid = "✓" if result['integrity_check'] else "✗"
        
        print(f"{name:<20} {shape_str:<15} {dtype_str:<10} {size_mb:<10.3f} {compress_ratio:<8.2f} {total_time:<10.2f} {valid:<6}")
        # every structure here round-trips (msgpack-numpy 0.4.8): a changed value, dtype or shape is a regression
        assert result['integrity_check'], (name, result)
    else:
        raise AssertionError(f"{name}: msgpack round-trip raised {result['error']}")

# Demonstrate advanced indexing preservation
print(f"\nAdvanced Indexing Test:")
base_array = np.arange(1000).reshape(20, 50)
fancy_indexed = base_array[[1, 5, 10, 15], :]  # Fancy indexing

try:
    serialized_fancy = msgpack.packb(fancy_indexed)
    deserialized_fancy = msgpack.unpackb(serialized_fancy, strict_map_key=False)
    fancy_preserved = np.array_equal(fancy_indexed, deserialized_fancy)
    print(f"Fancy indexing preserved: {fancy_preserved}")
    print(f"Original shape: {fancy_indexed.shape}, Deserialized shape: {deserialized_fancy.shape}")
except Exception as e:
    print(f"Fancy indexing error: {e}")
    raise

print("Complex array structures processing completed successfully!")

In [ ]:
# Scientific computing data pipelines and batch processing optimization
import io
import tempfile
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

print("=== msgpack-numpy Scientific Computing Pipeline ===")

class ScientificDataPipeline:
    def __init__(self, use_compression: bool = True):
        self.use_compression = use_compression
        self.processing_stats = {'files_processed': 0, 'total_time': 0}
    
    def process_dataset_batch(self, datasets: List[Dict[str, np.ndarray]]) -> List[bytes]:
        """Process a batch of scientific datasets for efficient storage/transmission"""
        
        processed_batches = []
        
        for i, dataset in enumerate(datasets):
            # Add metadata to dataset
            dataset_with_metadata = {
                'metadata': {
                    'batch_id': i,
                    'timestamp': time.time(),
                    'format_version': '1.0',
                    'arrays_count': len([v for v in dataset.values() if isinstance(v, np.ndarray)])
                },
                'data': dataset
            }
            
            # Serialize the complete dataset
            serialized = msgpack.packb(dataset_with_metadata)
            processed_batches.append(serialized)
        
        return processed_batches
    
    def simulate_experiment_data(self, n_experiments: int = 5) -> List[Dict[str, np.ndarray]]:
        """Simulate scientific experiment data with realistic structures"""
        
        experiments = []
        
        for exp_id in range(n_experiments):
            # Simulate different types of scientific data
            experiment = {
                'measurements': np.random.normal(100, 15, (1000, 50)).astype(np.float32),
                'timestamps': np.arange(1000).astype(np.float64),
                'conditions': np.random.choice(['A', 'B', 'C'], 1000).astype('U1'),
                'parameters': np.array([
                    [25.0, 1.2, 0.8],  # temperature, pH, concentration
                    [23.5, 1.1, 0.9],
                    [26.2, 1.3, 0.7]
                ]).astype(np.float32),
                'quality_flags': np.random.choice([True, False], 1000, p=[0.95, 0.05]),
                'spectra': np.random.exponential(2.0, (100, 2048)).astype(np.float32),
                'calibration_matrix': np.random.rand(10, 10).astype(np.float64)
            }
            experiments.append(experiment)
        
        return experiments
    
    def analyze_storage_efficiency(self, datasets: List[Dict[str, np.ndarray]]) -> Dict:
        """Analyze storage efficiency of msgpack-numpy vs alternatives"""
        
        import pickle
        
        # Calculate original memory usage
        total_memory = 0
        for dataset in datasets:
            for array in dataset.values():
                if isinstance(array, np.ndarray):
                    total_memory += array.nbytes
        
        # msgpack-numpy serialization
        start_time = time.time()
        msgpack_batches = self.process_dataset_batch(datasets)
        msgpack_time = time.time() - start_time
        msgpack_size = sum(len(batch) for batch in msgpack_batches)
        
        # Pickle serialization for comparison
        start_time = time.time()
        pickle_batches = []
        for dataset in datasets:
            pickle_data = pickle.dumps(dataset)
            pickle_batches.append(pickle_data)
        pickle_time = time.time() - start_time
        pickle_size = sum(len(batch) for batch in pickle_batches)
        
        return {
            'original_memory_mb': total_memory / (1024 * 1024),
            'msgpack_size_mb': msgpack_size / (1024 * 1024),
            'msgpack_time_ms': msgpack_time * 1000,
            'msgpack_compression_ratio': total_memory / msgpack_size,
            'pickle_size_mb': pickle_size / (1024 * 1024),
            'pickle_time_ms': pickle_time * 1000,
            'pickle_compression_ratio': total_memory / pickle_size,
            'size_advantage': pickle_size / msgpack_size,
            'speed_advantage': pickle_time / msgpack_time
        }

def parallel_processing_demo():
    """Demonstrate parallel processing of scientific datasets"""
    
    def process_single_dataset(dataset_info):
        """Process a single dataset (for parallel execution)"""
        dataset_id, dataset = dataset_info
        
        # Add some processing delay to simulate real work
        processed_data = {}
        for key, array in dataset.items():
            if isinstance(array, np.ndarray):
                # Simulate some scientific computation
                if array.dtype in [np.float32, np.float64]:
                    processed_data[f'{key}_normalized'] = (array - array.mean()) / array.std()
                processed_data[key] = array
            else:
                processed_data[key] = array
        
        # Serialize the processed dataset
        serialized = msgpack.packb(processed_data)
        
        return {
            'dataset_id': dataset_id,
            'serialized_size': len(serialized),
            'original_arrays': len([v for v in dataset.values() if isinstance(v, np.ndarray)]),
            'processed_arrays': len([v for v in processed_data.values() if isinstance(v, np.ndarray)])
        }
    
    pipeline = ScientificDataPipeline()
    datasets = pipeline.simulate_experiment_data(n_experiments=10)
    
    # Sequential processing
    start_time = time.time()
    sequential_results = [process_single_dataset((i, dataset)) for i, dataset in enumerate(datasets)]
    sequential_time = time.time() - start_time
    
    # Parallel processing
    start_time = time.time()
    with ThreadPoolExecutor(max_workers=4) as executor:
        parallel_results = list(executor.map(process_single_dataset, enumerate(datasets)))
    parallel_time = time.time() - start_time
    
    return {
        'datasets_count': len(datasets),
        'sequential_time': sequential_time,
        'parallel_time': parallel_time,
        'speedup_factor': sequential_time / parallel_time,
        'avg_dataset_size_kb': np.mean([r['serialized_size'] for r in parallel_results]) / 1024,
        'total_serialized_mb': sum(r['serialized_size'] for r in parallel_results) / (1024 * 1024)
    }

# Run scientific computing pipeline demonstration
pipeline = ScientificDataPipeline()

# Generate simulated experiment data
print("Generating simulated scientific experiment data...")
experiment_datasets = pipeline.simulate_experiment_data(n_experiments=8)

# Analyze storage efficiency
print("Analyzing storage efficiency...")
efficiency_analysis = pipeline.analyze_storage_efficiency(experiment_datasets)

print(f"\nStorage Efficiency Analysis:")
print(f"Original memory usage: {efficiency_analysis['original_memory_mb']:.2f} MB")
print(f"msgpack-numpy size: {efficiency_analysis['msgpack_size_mb']:.2f} MB")
print(f"msgpack-numpy compression: {efficiency_analysis['msgpack_compression_ratio']:.2f}x")
print(f"msgpack-numpy time: {efficiency_analysis['msgpack_time_ms']:.2f} ms")
print(f"\nComparison with Pickle:")
print(f"Pickle size: {efficiency_analysis['pickle_size_mb']:.2f} MB")
print(f"Size advantage: {efficiency_analysis['size_advantage']:.2f}x smaller")
print(f"Speed advantage: {efficiency_analysis['speed_advantage']:.2f}x faster")

# Test parallel processing
print(f"\nTesting parallel processing capabilities...")
parallel_results = parallel_processing_demo()

print(f"Processed {parallel_results['datasets_count']} datasets:")
print(f"Sequential time: {parallel_results['sequential_time']:.3f}s")
print(f"Parallel time: {parallel_results['parallel_time']:.3f}s")
print(f"Speedup factor: {parallel_results['speedup_factor']:.2f}x")
print(f"Average dataset size: {parallel_results['avg_dataset_size_kb']:.1f} KB")
print(f"Total serialized data: {parallel_results['total_serialized_mb']:.2f} MB")

print("Scientific computing data pipeline completed successfully!")

In [ ]:
# Distributed computing and cross-process array data exchange
import multiprocessing as mp
import os  # used by ArrayCacheManager.load_array (was missing -> cache demo silently vacuous)
import socket
import threading
from queue import Queue

print("=== msgpack-numpy Distributed Computing ===")

class DistributedArrayExchange:
    def __init__(self, port: int = 12345):
        self.port = port
        self.results_queue = Queue()
        
    def create_worker_process(self, worker_id: int, shared_data: Dict) -> mp.Process:
        """Create a worker process for distributed computation"""
        
        def worker_function(worker_id: int, input_data: Dict):
            """Worker function that processes arrays and returns results"""
            
            # Simulate distributed computation
            results = {}
            
            for name, array in input_data.items():
                if isinstance(array, np.ndarray):
                    # Different workers perform different operations
                    if worker_id % 3 == 0:
                        # Statistical operations
                        results[f'{name}_mean'] = np.mean(array)
                        results[f'{name}_std'] = np.std(array)
                    elif worker_id % 3 == 1:
                        # Linear algebra operations
                        if array.ndim == 2:
                            results[f'{name}_trace'] = np.trace(array)
                            results[f'{name}_det'] = np.linalg.det(array) if array.shape[0] == array.shape[1] else 0.0
                    else:
                        # Array transformations
                        results[f'{name}_max'] = np.max(array)
                        results[f'{name}_min'] = np.min(array)
            
            # Package results with worker metadata
            worker_result = {
                'worker_id': worker_id,
                'results': results,
                'processing_time': time.time(),
                'arrays_processed': len([v for v in input_data.values() if isinstance(v, np.ndarray)])
            }
            
            return worker_result
        
        process = mp.Process(target=worker_function, args=(worker_id, shared_data))
        return process
    
    def simulate_network_transfer(self, data_package: Dict) -> Tuple[bytes, float, Dict]:
        """Simulate network transfer of serialized array data"""
        
        # Serialize the data package
        start_time = time.time()
        serialized_data = msgpack.packb(data_package)
        serialize_time = time.time() - start_time
        
        # Simulate network latency and transfer
        transfer_start = time.time()
        # Simulate network delay proportional to data size
        simulated_latency = len(serialized_data) / (10 * 1024 * 1024)  # 10MB/s
        time.sleep(min(simulated_latency, 0.1))  # Cap at 100ms for demo
        
        # Deserialize on the "remote" end
        deserialized_data = msgpack.unpackb(serialized_data, strict_map_key=False)
        transfer_time = time.time() - transfer_start
        
        transfer_stats = {
            'serialized_size': len(serialized_data),
            'serialize_time': serialize_time,
            'transfer_time': transfer_time,
            'total_time': serialize_time + transfer_time,
            'throughput_mbps': (len(serialized_data) / (1024 * 1024)) / (serialize_time + transfer_time)
        }
        
        return serialized_data, transfer_time, transfer_stats

def distributed_computation_demo():
    """Demonstrate distributed array computation with msgpack-numpy"""
    
    # Create distributed data sets
    distributed_datasets = {}
    
    # Dataset 1: Large matrix computations
    distributed_datasets['matrix_data'] = {
        'large_matrix': np.random.rand(500, 500).astype(np.float32),
        'vector_data': np.random.rand(1000).astype(np.float64),
        'sparse_matrix': np.random.choice([0, 1], (200, 200), p=[0.9, 0.1]).astype(np.int32)
    }
    
    # Dataset 2: Time series data
    distributed_datasets['timeseries_data'] = {
        'signals': np.random.randn(10000, 20).astype(np.float32),
        'timestamps': np.arange(10000).astype(np.int64),
        'metadata': np.random.choice(['A', 'B', 'C'], 10000).astype('U1')
    }
    
    # Dataset 3: Scientific measurements
    distributed_datasets['measurement_data'] = {
        'sensor_readings': np.random.exponential(2.0, (5000, 16)).astype(np.float32),
        'calibration_data': np.random.rand(16, 16).astype(np.float64),
        'quality_mask': np.random.choice([True, False], 5000, p=[0.95, 0.05])
    }
    
    exchange = DistributedArrayExchange()
    
    # Simulate distributed processing
    distribution_results = {}
    
    for dataset_name, dataset in distributed_datasets.items():
        print(f"Processing {dataset_name}...")
        
        # Simulate network transfer
        serialized, transfer_time, stats = exchange.simulate_network_transfer(dataset)
        
        # Calculate original size
        original_size = sum(arr.nbytes for arr in dataset.values() if isinstance(arr, np.ndarray))
        
        distribution_results[dataset_name] = {
            'original_size_mb': original_size / (1024 * 1024),
            'serialized_size_mb': stats['serialized_size'] / (1024 * 1024),
            'compression_ratio': original_size / stats['serialized_size'],
            'serialize_time_ms': stats['serialize_time'] * 1000,
            'transfer_time_ms': stats['transfer_time'] * 1000,
            'total_time_ms': stats['total_time'] * 1000,
            'throughput_mbps': stats['throughput_mbps'],
            'arrays_count': len([v for v in dataset.values() if isinstance(v, np.ndarray)])
        }
    
    return distribution_results

class ArrayCacheManager:
    """Manage cached arrays using msgpack-numpy for efficient storage"""
    
    def __init__(self, cache_dir: str = '/tmp/msgpack_cache'):
        self.cache_dir = cache_dir
        self.cache_stats = {'hits': 0, 'misses': 0, 'stores': 0}
        
        # Create cache directory
        import os
        os.makedirs(cache_dir, exist_ok=True)
    
    def cache_key(self, array_id: str) -> str:
        """Generate cache file path"""
        return f"{self.cache_dir}/{array_id}.msgpack"
    
    def store_array(self, array_id: str, array: np.ndarray) -> bool:
        """Store array in cache using msgpack-numpy"""
        try:
            cache_path = self.cache_key(array_id)
            
            # Create cache entry with metadata
            cache_entry = {
                'array': array,
                'metadata': {
                    'shape': array.shape,
                    'dtype': str(array.dtype),
                    'cached_at': time.time(),
                    'array_id': array_id
                }
            }
            
            # Serialize and save
            serialized = msgpack.packb(cache_entry)
            with open(cache_path, 'wb') as f:
                f.write(serialized)
            
            self.cache_stats['stores'] += 1
            return True
            
        except Exception as e:
            print(f"Cache store error: {e}")
            return False
    
    def load_array(self, array_id: str) -> Union[np.ndarray, None]:
        """Load array from cache"""
        try:
            cache_path = self.cache_key(array_id)
            
            if not os.path.exists(cache_path):
                self.cache_stats['misses'] += 1
                return None
            
            # Load and deserialize
            with open(cache_path, 'rb') as f:
                serialized = f.read()
            
            cache_entry = msgpack.unpackb(serialized, strict_map_key=False)
            
            self.cache_stats['hits'] += 1
            return cache_entry['array']
            
        except Exception as e:
            print(f"Cache load error: {e}")
            self.cache_stats['misses'] += 1
            return None
    
    def get_cache_stats(self) -> Dict:
        """Get cache performance statistics"""
        total_requests = self.cache_stats['hits'] + self.cache_stats['misses']
        hit_rate = self.cache_stats['hits'] / total_requests if total_requests > 0 else 0
        
        return {
            **self.cache_stats,
            'total_requests': total_requests,
            'hit_rate': hit_rate
        }

# Run distributed computing demonstration
print("Running distributed computation simulation...")
distribution_results = distributed_computation_demo()

print(f"\nDistributed Computing Results:")
print(f"{'Dataset':<20} {'Original(MB)':<12} {'Serialized(MB)':<14} {'Compression':<12} {'Transfer(ms)':<12} {'Throughput':<12}")
print("-" * 85)

for dataset, stats in distribution_results.items():
    print(f"{dataset:<20} {stats['original_size_mb']:<12.2f} {stats['serialized_size_mb']:<14.2f} "
          f"{stats['compression_ratio']:<12.2f} {stats['transfer_time_ms']:<12.1f} {stats['throughput_mbps']:<12.1f}")

# Test array caching system
print(f"\nTesting array caching system...")
cache_manager = ArrayCacheManager()

# Store some test arrays
test_arrays = {
    'model_weights': np.random.rand(1000, 500).astype(np.float32),
    'training_data': np.random.randn(5000, 100).astype(np.float32),
    'results_cache': np.random.rand(200, 200).astype(np.float64)
}

for array_id, array in test_arrays.items():
    cache_manager.store_array(array_id, array)

# Test cache retrieval
retrieved_arrays = {}
for array_id in test_arrays.keys():
    retrieved = cache_manager.load_array(array_id)
    if retrieved is not None:
        retrieved_arrays[array_id] = retrieved

cache_stats = cache_manager.get_cache_stats()

print(f"Cache Performance:")
print(f"Arrays stored: {cache_stats['stores']}")
print(f"Cache hits: {cache_stats['hits']}")
print(f"Cache misses: {cache_stats['misses']}")
print(f"Hit rate: {cache_stats['hit_rate']*100:.1f}%")
print(f"Retrieved arrays: {len(retrieved_arrays)}/{len(test_arrays)}")

print("Distributed computing and data exchange completed successfully!")

In [ ]:
# Production optimization patterns and advanced msgpack-numpy usage
import hashlib
import zlib
from pathlib import Path

print("=== msgpack-numpy Production Optimization ===")

class OptimizedArraySerializer:
    """Production-optimized array serializer with advanced features"""
    
    def __init__(self, 
                 enable_compression: bool = True,
                 enable_checksums: bool = True,
                 chunk_size: int = 64 * 1024):  # 64KB chunks
        self.enable_compression = enable_compression
        self.enable_checksums = enable_checksums
        self.chunk_size = chunk_size
        self.stats = {'serializations': 0, 'deserializations': 0, 'errors': 0}
    
    def serialize_with_metadata(self, data: Dict[str, np.ndarray], metadata: Dict = None) -> bytes:
        """Serialize arrays with comprehensive metadata and optimization"""
        
        package = {
            'format_version': '2.0',
            'timestamp': time.time(),
            'compression': self.enable_compression,
            'checksums': self.enable_checksums,
            'metadata': metadata or {},
            'arrays': {}
        }
        
        # Process each array
        for name, array in data.items():
            if isinstance(array, np.ndarray):
                array_info = {
                    'data': array,
                    'original_shape': array.shape,
                    'original_dtype': str(array.dtype),
                    'memory_layout': array.flags.c_contiguous
                }
                
                # Add checksum if enabled
                if self.enable_checksums:
                    array_bytes = array.tobytes()
                    array_info['checksum'] = hashlib.md5(array_bytes).hexdigest()
                
                package['arrays'][name] = array_info
            else:
                package['arrays'][name] = data[name]
        
        # Serialize with msgpack-numpy
        serialized = msgpack.packb(package)
        
        # Apply compression if enabled
        if self.enable_compression:
            compressed = zlib.compress(serialized, level=6)
            final_data = {
                'compressed': True,
                'original_size': len(serialized),
                'data': compressed
            }
            serialized = msgpack.packb(final_data)
        
        self.stats['serializations'] += 1
        return serialized
    
    def deserialize_with_validation(self, serialized_data: bytes) -> Tuple[Dict, Dict]:
        """Deserialize with full validation and error checking"""
        
        try:
            # Check if data is compressed
            temp_package = msgpack.unpackb(serialized_data, strict_map_key=False)
            
            if isinstance(temp_package, dict) and temp_package.get('compressed', False):
                # Decompress the data
                compressed_data = temp_package['data']
                decompressed = zlib.decompress(compressed_data)
                package = msgpack.unpackb(decompressed, strict_map_key=False)
            else:
                package = temp_package
            
            # Extract metadata and arrays
            arrays = {}
            validation_results = {'passed': [], 'failed': []}
            
            for name, array_info in package.get('arrays', {}).items():
                if isinstance(array_info, dict) and 'data' in array_info:
                    array = array_info['data']
                    
                    # Validate checksum if available
                    if self.enable_checksums and 'checksum' in array_info:
                        current_checksum = hashlib.md5(array.tobytes()).hexdigest()
                        if current_checksum == array_info['checksum']:
                            validation_results['passed'].append(name)
                        else:
                            validation_results['failed'].append(name)
                    
                    arrays[name] = array
                else:
                    arrays[name] = array_info
            
            if validation_results['failed']:
                # a checksum mismatch is corrupt data: refuse it (re-raised below as RuntimeError), never return it
                raise ValueError(f"Checksum validation failed for {validation_results['failed']}")
            
            metadata = {
                'format_version': package.get('format_version'),
                'timestamp': package.get('timestamp'),
                'validation': validation_results,
                'original_metadata': package.get('metadata', {})
            }
            
            self.stats['deserializations'] += 1
            return arrays, metadata
            
        except Exception as e:
            self.stats['errors'] += 1
            raise RuntimeError(f"Deserialization failed: {e}")
    
    def get_stats(self) -> Dict:
        """Get serializer performance statistics"""
        return self.stats.copy()

def production_workflow_demo():
    """Demonstrate complete production workflow"""
    
    # Initialize optimized serializer
    serializer = OptimizedArraySerializer(
        enable_compression=True,
        enable_checksums=True
    )
    
    # Create realistic production data
    production_data = {
        'model_weights': np.random.randn(2000, 1500).astype(np.float32),
        'feature_matrix': np.random.rand(5000, 200).astype(np.float32),
        'labels': np.random.randint(0, 10, 5000).astype(np.int32),
        'embeddings': np.random.randn(1000, 128).astype(np.float16),
        'metadata_flags': np.random.choice([True, False], 5000, p=[0.8, 0.2])
    }
    
    metadata = {
        'experiment_id': 'prod_exp_001',
        'model_version': '2.1.0',
        'training_date': '2023-12-01',
        'validation_score': 0.945
    }
    
    # Serialize with optimization
    print("Serializing production data...")
    start_time = time.time()
    serialized = serializer.serialize_with_metadata(production_data, metadata)
    serialize_time = time.time() - start_time
    
    # Calculate statistics
    original_size = sum(arr.nbytes for arr in production_data.values() if isinstance(arr, np.ndarray))
    compression_ratio = original_size / len(serialized)
    
    print(f"Serialization completed:")
    print(f"  Original size: {original_size / (1024*1024):.1f} MB")
    print(f"  Serialized size: {len(serialized) / (1024*1024):.1f} MB")
    print(f"  Compression ratio: {compression_ratio:.2f}x")
    print(f"  Serialization time: {serialize_time:.3f}s")
    print(f"  Throughput: {(original_size/(1024*1024))/serialize_time:.1f} MB/s")
    
    # Deserialize with validation
    print(f"\nDeserializing and validating...")
    start_time = time.time()
    deserialized_data, deserialized_metadata = serializer.deserialize_with_validation(serialized)
    deserialize_time = time.time() - start_time
    
    # Validation results
    validation_passed = len(deserialized_metadata['validation']['passed'])
    validation_failed = len(deserialized_metadata['validation']['failed'])
    
    print(f"Deserialization completed:")
    print(f"  Deserialization time: {deserialize_time:.3f}s")
    print(f"  Validation passed: {validation_passed}")
    print(f"  Validation failed: {validation_failed}")
    print(f"  Data integrity: {'✓' if validation_failed == 0 else '✗'}")
    
    # Test array equality
    arrays_equal = all(
        np.array_equal(production_data[name], deserialized_data[name])
        for name in production_data.keys()
        if isinstance(production_data[name], np.ndarray)
    )
    
    print(f"  Arrays identical: {'✓' if arrays_equal else '✗'}")
    # every array carries a checksum, so every one must have been checked, and come back bit-identical
    assert validation_passed == len(production_data) and validation_failed == 0, deserialized_metadata['validation']
    assert arrays_equal, "a round-tripped array differs from its original"
    
    return {
        'original_size_mb': original_size / (1024*1024),
        'serialized_size_mb': len(serialized) / (1024*1024),
        'compression_ratio': compression_ratio,
        'serialize_time': serialize_time,
        'deserialize_time': deserialize_time,
        'validation_passed': validation_passed,
        'validation_failed': validation_failed,
        'data_integrity': arrays_equal
    }

# Run production optimization demonstration
workflow_results = production_workflow_demo()

print(f"\nProduction Workflow Summary:")
print(f"Data size reduced: {workflow_results['original_size_mb']:.1f} MB → {workflow_results['serialized_size_mb']:.1f} MB")
print(f"Space savings: {(1 - 1/workflow_results['compression_ratio'])*100:.1f}%")
print(f"Total processing time: {workflow_results['serialize_time'] + workflow_results['deserialize_time']:.3f}s")
print(f"Data integrity maintained: {workflow_results['data_integrity']}")

# Performance recommendations
print(f"\nProduction Recommendations:")
if workflow_results['compression_ratio'] > 2.0:
    print("✓ Excellent compression achieved - suitable for network transfer")
else:
    print("• Consider enabling compression for better network efficiency")

if workflow_results['serialize_time'] + workflow_results['deserialize_time'] < 1.0:
    print("✓ Fast serialization/deserialization - suitable for real-time applications")
else:
    print("• Consider optimizing for latency-sensitive applications")

if workflow_results['validation_failed'] == 0:
    print("✓ Perfect data integrity - production ready")
else:
    print("⚠ Data integrity issues detected - investigate checksums")

print("Production optimization patterns completed successfully!")